# KITTI 3D LiDAR training

Chạy lần lượt từ trên xuống. Khi Colab ngắt, chạy lại cell **Train / resume**.

In [ ]:
from pathlib import Path
import json
import os

BRANCH = "integrated_P12"
VARIANT = "C0"  # A0-A4, B0-B3, C0-C2
RUN_NAME = None
RESUME = False
SEED = 42
EPOCHS = 50
PRECISION = "bf16"  # fp32, fp16, bf16
PHYSICAL_BATCH_SIZE = 16
ACCUMULATION_STEPS = 1
RUN_CHECKS = True
RUN_SMOKE_TEST = True
RUN_EVALUATION = True
NUM_WORKERS = max(0, min(6, (os.cpu_count() or 1) - 1))

VARIANT_CONFIGS = {
    "A0": "configs/kitti/kitti_uwag_coordatt_aug.json",
    "A1": "configs/kitti/mobilebev/a1_legacy35_center3d.json",
    "A2": "configs/kitti/mobilebev/a2_rich8_center3d.json",
    "A3": "configs/kitti/mobilebev/a3_legacy35_sgfpn_center3d.json",
    "A4": "configs/kitti/mobilebev/a4_rich8_sgfpn_center3d.json",
    "B0": "configs/kitti/probgeo_uq/b0_deterministic.json",
    "B1": "configs/kitti/probgeo_uq/b1_gwd.json",
    "B2": "configs/kitti/probgeo_uq/b2_heteroscedastic.json",
    "B3": "configs/kitti/probgeo_uq/b3_probgeo_uq.json",
    "C0": "configs/kitti/probgeo_uq/c0_a1_probgeo_uq.json",
    "C1": "configs/kitti/probgeo_uq/c1_a3_probgeo_uq.json",
    "C2": "configs/kitti/probgeo_uq/c2_a4_probgeo_uq.json",
}
if VARIANT not in VARIANT_CONFIGS:
    raise ValueError(f"Variant phải là một trong: {', '.join(VARIANT_CONFIGS)}")
CONFIG_PATH = VARIANT_CONFIGS[VARIANT]

REPOSITORY_URL = "https://github.com/danhyoyo/Lidar.git"
REPO_DIR = Path("/content/Lidar")
KITTI_TAR_ROOT = Path("/content/drive/MyDrive/KITTI_DATASET_ZIP")
RAW_KITTI_ROOT = Path("/content/KITTI_DATASET")
PROCESSED_DATASET_DIR = REPO_DIR / "data/kitti/processed"
ARTIFACT_ROOT = Path("/content/drive/MyDrive/lidar_training_artifacts")

## Setup

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
%cd /content
!test -d "{REPO_DIR}" || git clone "{REPOSITORY_URL}" "{REPO_DIR}"
if _exit_code:
    raise RuntimeError("Clone repository thất bại")
%cd {REPO_DIR}
!git fetch --prune origin "+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"
if _exit_code:
    raise RuntimeError(f"Không fetch được branch {BRANCH}")
!git checkout --detach "origin/{BRANCH}"
if _exit_code:
    raise RuntimeError(f"Không checkout được branch {BRANCH}")
!python3 -m pip install -q shapely tqdm
if _exit_code:
    raise RuntimeError("Cài dependency thất bại")

import torch

COMMIT = !git rev-parse HEAD
COMMIT = COMMIT[0]
if not torch.cuda.is_available():
    raise RuntimeError("Hãy bật GPU trong Runtime > Change runtime type")
if PRECISION == "bf16" and not torch.cuda.is_bf16_supported():
    raise RuntimeError("GPU không hỗ trợ BF16; hãy dùng PRECISION='fp16'")
print(f"Branch: {BRANCH} @ {COMMIT}")
print(f"Variant: {VARIANT} | Config: {CONFIG_PATH}")
print(f"PyTorch: {torch.__version__} | GPU: {torch.cuda.get_device_name(0)}")

## Dataset

In [ ]:
archives = {
    "velodyne": ("*.bin", 7481),
    "label_2": ("*.txt", 7481),
    "calib": ("*.txt", 7481),
}
RAW_KITTI_ROOT.mkdir(parents=True, exist_ok=True)

for folder, (pattern, expected) in archives.items():
    archive = KITTI_TAR_ROOT / f"{folder}.tar"
    extracted = RAW_KITTI_ROOT / "training" / folder
    if sum(1 for _ in extracted.glob(pattern)) != expected:
        if not archive.is_file():
            raise FileNotFoundError(archive)
        print(f"Extracting: {archive}")
        archive_bytes = archive.stat().st_size
        !set -o pipefail; python3 -m tqdm --bytes --total {archive_bytes} --desc "Extract {folder}" < "{archive}" | tar --no-same-owner -xf - -C "{RAW_KITTI_ROOT}"
        if _exit_code:
            raise RuntimeError(f"Giải nén thất bại: {archive}")
    actual = sum(1 for _ in extracted.glob(pattern))
    if actual != expected:
        raise RuntimeError(f"{folder}: có {actual}/{expected} file")

pointcloud_dir = PROCESSED_DATASET_DIR / "pointcloud"
label_dir = PROCESSED_DATASET_DIR / "label"
dataset_ready = (
    sum(1 for _ in pointcloud_dir.glob("*.bin")) == 7481
    and sum(1 for _ in label_dir.glob("*.txt")) == 7481
    and (PROCESSED_DATASET_DIR / "train.txt").is_file()
    and (PROCESSED_DATASET_DIR / "val.txt").is_file()
)
if not dataset_ready:
    print("Preparing KITTI dataset...")
    !python3 -u tools/kitti_training_pipeline/prepare_kitti.py --kitti-root "{RAW_KITTI_ROOT}" --output-root "{PROCESSED_DATASET_DIR}" --config-output "{REPO_DIR / 'data/kitti/generated_kitti.json'}" --train-ids "{REPO_DIR / 'splits/kitti/train.txt'}" --val-ids "{REPO_DIR / 'splits/kitti/val.txt'}" --pointcloud-mode symlink --overwrite
    if _exit_code:
        raise RuntimeError("Chuẩn bị KITTI thất bại")

pointcloud_count = sum(1 for _ in pointcloud_dir.glob("*.bin"))
label_count = sum(1 for _ in label_dir.glob("*.txt"))
if pointcloud_count != 7481 or label_count != 7481:
    raise RuntimeError(f"Processed dataset thiếu file: {pointcloud_count} point clouds, {label_count} labels")
print(f"Dataset: {PROCESSED_DATASET_DIR}")
print(f"Frames: {pointcloud_count} | train: 5984 | val: 1497")

## Checks / smoke test

In [ ]:
from uuid import uuid4

CONFIG = (REPO_DIR / CONFIG_PATH).resolve()
if not CONFIG.is_file():
    raise FileNotFoundError(CONFIG)

CONFIG_DATA = json.loads(CONFIG.read_text(encoding="utf-8"))
EFFECTIVE_BATCH_SIZE = PHYSICAL_BATCH_SIZE * ACCUMULATION_STEPS

if RUN_NAME is None:
    RUN_NAME = f"{VARIANT.lower()}_seed{SEED}_eb{EFFECTIVE_BATCH_SIZE}"

print(f"Config: {CONFIG}")
print(f"Run name: {RUN_NAME}")

if RUN_CHECKS:
    print("Checks: running tests/test_mobile_bev.py")
    !MPLCONFIGDIR=/tmp/lidar-mpl python3 tests/test_mobile_bev.py

    if _exit_code:
        raise RuntimeError("Code checks thất bại")

    print("Checks: PASS")

if RUN_SMOKE_TEST:
    SMOKE_ROOT = Path("/content/lidar_smoke_artifacts")
    SMOKE_RUN_NAME = f"{RUN_NAME}_resume_{uuid4().hex[:8]}"
    SMOKE_CHECKPOINT = SMOKE_ROOT / SMOKE_RUN_NAME / "checkpoints" / "last.pt"

    print(f"Smoke epoch 1: {VARIANT}")
    !python3 -u tools/kitti_training_pipeline/train.py \
      --config "{CONFIG}" \
      --detector-root detector \
      --output-root "{SMOKE_ROOT}" \
      --run-name "{SMOKE_RUN_NAME}" \
      --device cuda \
      --precision "{PRECISION}" \
      --seed {SEED} \
      --epochs 1 \
      --physical-batch-size {PHYSICAL_BATCH_SIZE} \
      --accumulation-steps {ACCUMULATION_STEPS} \
      --max-train-batches 8 \
      --max-val-batches 4 \
      --num-workers {NUM_WORKERS}

    if _exit_code:
        raise RuntimeError("Smoke epoch 1 thất bại")
    if not SMOKE_CHECKPOINT.is_file():
        raise FileNotFoundError(SMOKE_CHECKPOINT)

    first_state = torch.load(
        SMOKE_CHECKPOINT,
        map_location="cpu",
        weights_only=False,
    )
    if first_state.get("epoch") != 1:
        raise RuntimeError(
            f"Checkpoint smoke phải ở epoch 1, nhận được {first_state.get('epoch')}"
        )

    first_scheduler_epoch = first_state["scheduler_state_dict"]["last_epoch"]
    print(f"Smoke epoch 1: PASS | checkpoint: {SMOKE_CHECKPOINT}")
    print("Smoke resume: epoch 1 → epoch 2")

    !python3 -u tools/kitti_training_pipeline/train.py \
      --config "{CONFIG}" \
      --detector-root detector \
      --output-root "{SMOKE_ROOT}" \
      --run-name "{SMOKE_RUN_NAME}" \
      --device cuda \
      --precision "{PRECISION}" \
      --seed {SEED} \
      --epochs 2 \
      --physical-batch-size {PHYSICAL_BATCH_SIZE} \
      --accumulation-steps {ACCUMULATION_STEPS} \
      --max-train-batches 8 \
      --max-val-batches 4 \
      --num-workers {NUM_WORKERS} \
      --resume "{SMOKE_CHECKPOINT}"

    if _exit_code:
        raise RuntimeError("Smoke resume epoch 2 thất bại")

    resumed_state = torch.load(
        SMOKE_CHECKPOINT,
        map_location="cpu",
        weights_only=False,
    )
    if resumed_state.get("epoch") != 2:
        raise RuntimeError(
            f"Resume phải tạo checkpoint epoch 2, nhận được "
            f"{resumed_state.get('epoch')}"
        )

    resumed_scheduler_epoch = resumed_state["scheduler_state_dict"]["last_epoch"]
    if resumed_scheduler_epoch != first_scheduler_epoch + 1:
        raise RuntimeError(
            "Scheduler không tiếp tục đúng: "
            f"{first_scheduler_epoch} → {resumed_scheduler_epoch}"
        )

    print(
        "Smoke resume: PASS | "
        f"epoch 1 → 2 | scheduler "
        f"{first_scheduler_epoch} → {resumed_scheduler_epoch}"
    )


## Train / resume

In [ ]:
RUN_DIR = ARTIFACT_ROOT / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
LAST_CHECKPOINT = CHECKPOINT_DIR / "last.pt"
TRAIN_LOG = RUN_DIR / "train.log"
RUN_DIR.mkdir(parents=True, exist_ok=True)


if RESUME and not LAST_CHECKPOINT.is_file():
    raise FileNotFoundError(f"RESUME=True nhưng không có {LAST_CHECKPOINT}")
if not RESUME and LAST_CHECKPOINT.is_file():
    raise FileExistsError(f"Run đã tồn tại; đặt RESUME=True hoặc đổi RUN_NAME: {RUN_DIR}")
resume_epoch = 0
RUN_INFO = {
    "variant": VARIANT,
    "seed": SEED,
    "precision": PRECISION,
    "epochs": EPOCHS,
    "physical_batch_size": PHYSICAL_BATCH_SIZE,
    "accumulation_steps": ACCUMULATION_STEPS,
}
(RUN_DIR / "run.json").write_text(
    json.dumps(RUN_INFO, indent=2) + "\n", encoding="utf-8"
)

if RESUME:
    state = torch.load(LAST_CHECKPOINT, map_location="cpu", weights_only=False)
    resume_epoch = int(state.get("epoch", 0))
RESUME_ARGUMENT = f'--resume "{LAST_CHECKPOINT}"' if RESUME else ""

print(f"Run: {RUN_DIR}")
print(f"Mode: {'resume' if RESUME else 'new'} | epoch: {resume_epoch}/{EPOCHS}")
print(f"Batch: {PHYSICAL_BATCH_SIZE} x {ACCUMULATION_STEPS} = {EFFECTIVE_BATCH_SIZE} | precision: {PRECISION}")
if resume_epoch >= EPOCHS:
    print("Training đã hoàn tất.")
else:
    !set -o pipefail; python3 -u tools/kitti_training_pipeline/train.py --config "{CONFIG}" --detector-root detector --output-root "{ARTIFACT_ROOT}" --run-name "{RUN_NAME}" --device cuda --precision "{PRECISION}" --seed {SEED} --epochs {EPOCHS} --physical-batch-size {PHYSICAL_BATCH_SIZE} --accumulation-steps {ACCUMULATION_STEPS} --num-workers {NUM_WORKERS} {RESUME_ARGUMENT} 2>&1 | tee -a "{TRAIN_LOG}"
    if _exit_code:
        raise RuntimeError(f"Training thất bại với exit code {_exit_code}")

## Select checkpoint / evaluate

In [ ]:
RESOLVED_CONFIG = RUN_DIR / "config.resolved.json"
if not RESOLVED_CONFIG.is_file():
    raise FileNotFoundError(RESOLVED_CONFIG)
selection_split = REPO_DIR / CONFIG_DATA["val"]["data"]
is_center3d = CONFIG_DATA["model"].get("box_encoding", "bev") == "center3d"

if is_center3d:
    SELECTED_DIR = RUN_DIR / "selected_3d"
    print(f"Selecting checkpoint on: {selection_split}")
    !python3 -u tools/kitti_training_pipeline/select_checkpoint.py --checkpoint-dir "{CHECKPOINT_DIR}" --config "{RESOLVED_CONFIG}" --detector-root detector --kitti-root "{RAW_KITTI_ROOT}" --split "{selection_split}" --output-dir "{SELECTED_DIR}" --device cuda
    if _exit_code:
        raise RuntimeError("Chọn checkpoint thất bại")
    CHECKPOINT = SELECTED_DIR / "best.pt"
else:
    CHECKPOINT = RUN_DIR / "selected" / "best.pt"
if not CHECKPOINT.is_file():
    raise FileNotFoundError(CHECKPOINT)
print(f"Checkpoint: {CHECKPOINT}")

if RUN_EVALUATION:
    has_uq = CONFIG_DATA["model"].get("predict_log_variance", False)
    evaluation_splits = [("calibration", selection_split), ("test", REPO_DIR / "splits/kitti/uq_test.txt")] if has_uq else [("validation", selection_split)]
    outputs = {}
    for name, split in evaluation_splits:
        output = RUN_DIR / f"evaluation_{name}.json"
        print(f"Evaluating {name}: {split}")
        !python3 -u tools/kitti_training_pipeline/evaluate_kitti_bev.py --name "{RUN_NAME}_{name}" --backend pytorch --model "{CHECKPOINT}" --config "{RESOLVED_CONFIG}" --detector-root detector --kitti-root "{RAW_KITTI_ROOT}" --split "{split}" --output "{output}" --device cuda --warmup-frames 10
        if _exit_code:
            raise RuntimeError(f"Đánh giá {name} thất bại")
        outputs[name] = output
        print(f"Evaluation: {output}")
    if has_uq:
        uncertainty_output = RUN_DIR / "uncertainty_test.json"
        !python3 -u tools/kitti_training_pipeline/evaluate_uncertainty.py --predictions "{outputs['test'].with_suffix('.predictions.npz')}" --config "{RESOLVED_CONFIG}" --kitti-root "{RAW_KITTI_ROOT}" --split "{REPO_DIR / 'splits/kitti/uq_test.txt'}" --calibration-predictions "{outputs['calibration'].with_suffix('.predictions.npz')}" --calibration-split "{selection_split}" --output "{uncertainty_output}"
        if _exit_code:
            raise RuntimeError("Đánh giá uncertainty thất bại")
        print(f"Uncertainty: {uncertainty_output}")